<a href="https://colab.research.google.com/github/sanjananagendra16-cpu/SIH2026/blob/main/SIH_Crop_Disease.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import tensorflow as tf

print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

2.20.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
import tensorflow_datasets as tfds

print(tfds)
print(tfds.__file__)
print(hasattr(tfds, "load"))

<module 'tensorflow_datasets' from '/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/__init__.py'>
/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/__init__.py
False


In [5]:
!git clone -q https://github.com/spMohanty/PlantVillage-Dataset.git
print("PlantVillage downloaded!")

PlantVillage downloaded!


In [6]:
import os

path = "/content/PlantVillage-Dataset/raw/color"

print(os.listdir(path)[:10])
print("Number of classes:", len(os.listdir(path)))

['Potato___healthy', 'Apple___Black_rot', 'Tomato___healthy', 'Soybean___healthy', 'Apple___healthy', 'Apple___Cedar_apple_rust', 'Pepper,_bell___healthy', 'Tomato___Tomato_mosaic_virus', 'Tomato___Target_Spot', 'Squash___Powdery_mildew']
Number of classes: 38


In [8]:
import os
import shutil

source = "/content/PlantVillage-Dataset/raw/color"
target = "/content/potato_dataset"

classes = [
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy"
]

# Start fresh
if os.path.exists(target):
    shutil.rmtree(target)

os.makedirs(target)

# Use the same number from every class
IMAGES_PER_CLASS = 150

for cls in classes:
    src = os.path.join(source, cls)
    dst = os.path.join(target, cls)
    os.makedirs(dst)

    files = os.listdir(src)[:IMAGES_PER_CLASS]

    for file in files:
        shutil.copy(
            os.path.join(src, file),
            os.path.join(dst, file)
        )

print("Balanced dataset prepared!\n")

for cls in classes:
    print(cls, ":", len(os.listdir(os.path.join(target, cls))))

Balanced dataset prepared!

Potato___Early_blight : 150
Potato___Late_blight : 150
Potato___healthy : 150


In [9]:
import tensorflow as tf

IMG_SIZE = (160, 160)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/potato_dataset",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/potato_dataset",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names

print("Classes:", class_names)

Found 450 files belonging to 3 classes.
Using 360 files for training.
Found 450 files belonging to 3 classes.
Using 90 files for validation.
Classes: ['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']


In [10]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

model = models.Sequential([
    layers.Rescaling(1./127.5, offset=-1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(3, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)

In [11]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 49s 3s/step - accuracy: 0.6139 - loss: 0.8499 - val_accuracy: 0.8667 - val_loss: 0.4508
Epoch 2/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.8778 - loss: 0.4006 - val_accuracy: 0.9000 - val_loss: 0.2679
Epoch 3/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9056 - loss: 0.2630 - val_accuracy: 0.9222 - val_loss: 0.2214
Epoch 4/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9333 - loss: 0.1861 - val_accuracy: 0.9333 - val_loss: 0.1918
Epoch 5/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.9444 - loss: 0.1489 - val_accuracy: 0.9333 - val_loss: 0.1790


In [12]:
import numpy as np
import tensorflow as tf

class_names = [
    "Potato Early Blight",
    "Potato Late Blight",
    "Potato Healthy"
]

def predict_disease(image):
    # Resize image
    img = tf.image.resize(image, (160, 160))

    # Convert to array
    img = tf.cast(img, tf.float32) / 255.0

    # Add batch dimension
    img = tf.expand_dims(img, 0)

    # Prediction
    predictions = model.predict(img, verbose=0)[0]

    index = np.argmax(predictions)
    confidence = predictions[index] * 100

    disease = class_names[index]

    return disease, confidence

In [13]:
from google.colab import files
from PIL import Image

uploaded = files.upload()

for filename in uploaded.keys():
    image = Image.open(filename).convert("RGB")
    image = np.array(image)

    disease, confidence = predict_disease(image)

    print("🌱 CROP DISEASE DETECTION")
    print("-------------------------")
    print("Prediction:", disease)
    print(f"Confidence: {confidence:.2f}%")

Saving Screenshot_18-9-2026_223456_bpb-us-e1.wpmucdn.com.jpeg to Screenshot_18-9-2026_223456_bpb-us-e1.wpmucdn.com.jpeg
🌱 CROP DISEASE DETECTION
-------------------------
Prediction: Potato Late Blight
Confidence: 58.89%


In [14]:
import numpy as np
import tensorflow as tf

class_names = [
    "Potato Early Blight",
    "Potato Late Blight",
    "Potato Healthy"
]

def predict_disease(image):

    # Resize image
    img = tf.image.resize(image, (160, 160))

    # Add batch dimension
    img = tf.expand_dims(img, axis=0)

    # Predict
    predictions = model.predict(img, verbose=0)[0]

    index = np.argmax(predictions)
    confidence = predictions[index] * 100

    return class_names[index], confidence

In [15]:
import gradio as gr

def app_predict(image):

    disease, confidence = predict_disease(image)

    if disease == "Potato Early Blight":
        advice = """
Management Guidance:
• Remove severely affected leaves.
• Maintain proper field sanitation.
• Avoid unnecessary leaf wetness.
• Monitor nearby plants regularly.
"""

    elif disease == "Potato Late Blight":
        advice = """
Management Guidance:
• Inspect surrounding plants immediately.
• Remove severely affected plant material.
• Avoid excessive moisture on foliage.
• Follow locally recommended agricultural practices.
"""

    else:
        advice = """
Management Guidance:
• Crop appears healthy.
• Continue regular monitoring.
• Maintain proper irrigation and field sanitation.
"""

    return f"""
🌱 CROP DISEASE DETECTION

Disease/Condition: {disease}

Confidence: {confidence:.2f}%

{advice}
"""


demo = gr.Interface(
    fn=app_predict,
    inputs=gr.Image(
        type="numpy",
        label="Upload Potato Leaf Image"
    ),
    outputs=gr.Textbox(
        label="AI Analysis"
    ),
    title="🌾 AI-Based Crop Disease Detection",
    description="Upload a potato leaf image for AI-based disease detection and management guidance."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0985ccb52394b3bf9c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
